# 読売新聞の記事検索結果を取得する

読売新聞オンラインで「熊本地震」を検索し、2026年7月27日以降に公開された記事のうち、`#熊本地震`タグが付いた記事だけのタイトル・本文・公開日時・URLを取得します。

- 読売IDでログインし、契約上閲覧できる本文のみ取得します。アクセス制限の回避は行いません。
- 検索一覧のタグを先に確認し、タグなし記事の本文ページにはアクセスしません。
- 短時間に大量アクセスしないよう待機時間を設けています。
- 実行前に利用規約・著作権・robots.txtを確認し、取得データは許可された範囲で利用してください。


In [1]:
# 初回だけ実行してください
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "requests", "beautifulsoup4", "pandas", "playwright"
])



[notice] A new release of pip is available: 24.3.1 -> 26.2
[notice] To update, run: pip install --upgrade pip


0

In [2]:
import json
import time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.yomiuri.co.jp"
SEARCH_URL = f"{BASE_URL}/web-search/"
LOGIN_URL = f"{BASE_URL}/member/login/"
KEYWORD = "熊本地震"
REQUIRED_TAG = "#熊本地震"
START_DATE = pd.Timestamp("2026-07-27", tz="Asia/Tokyo")

REQUEST_INTERVAL = 1.5
TIMEOUT = 30

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0 Safari/537.36"
    ),
    "Accept-Language": "ja,en-US;q=0.8,en;q=0.6",
})


## ログイン情報の入力

IDとパスワードは変数にだけ保持され、ノートブックファイルには保存されません。パスワードは入力中も画面に表示されません。


In [3]:
from getpass import getpass

YOMIURI_ID = input("読売ID（メールアドレス）: ").strip()
YOMIURI_PASSWORD = getpass("パスワード: ")


In [4]:
from playwright.async_api import TimeoutError as PlaywrightTimeoutError
from playwright.async_api import async_playwright


async def login_to_yomiuri(requests_session, yomiuri_id, password):
    """Chromeで読売IDにログインし、Cookieをrequestsへ渡す。"""
    if not yomiuri_id or not password:
        raise ValueError("読売IDとパスワードを入力してください。")

    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(channel="chrome", headless=False)
        context = await browser.new_context(locale="ja-JP")
        page = await context.new_page()
        await page.goto(LOGIN_URL, wait_until="domcontentloaded", timeout=60_000)

        username = page.locator(
            'input[autocomplete="username"], input[type="email"], '
            'input[name*="mail" i], input[name*="login" i], input[type="text"]'
        ).first
        await username.wait_for(state="visible", timeout=30_000)
        await username.fill(yomiuri_id)

        password_input = page.locator('input[type="password"]').first
        if not await password_input.is_visible():
            await page.locator('button[type="submit"], input[type="submit"]').first.click()
            await password_input.wait_for(state="visible", timeout=30_000)

        await password_input.fill(password)

        # パスワード欄と同じフォームの送信ボタンを優先する。
        login_form = password_input.locator("xpath=ancestor::form[1]")
        submit = login_form.locator(
            'button[type="submit"], input[type="submit"]'
        ).first
        if await submit.count() == 0:
            submit = page.locator(
                'button[type="submit"], input[type="submit"]'
            ).first

        before_submit_url = page.url
        try:
            # 読売トップへの遷移後、広告等の読込待ちでTimeoutになるのを防ぐ。
            await submit.click(timeout=30_000, no_wait_after=True)
        except PlaywrightTimeoutError:
            # 実際には遷移済みならログイン処理を続行する。
            if page.url == before_submit_url:
                raise
            print(f"次のログイン画面へ遷移済みです: {page.url}")

        await page.wait_for_timeout(2_000)

        # 読売側の中間画面に「ログインの方はこちら」があれば自動クリックを試す。
        continue_button = page.locator(
            'a:has-text("ログインの方はこちら"), '
            'button:has-text("ログインの方はこちら"), '
            'input[type="submit"][value*="ログイン"]'
        ).first
        if await continue_button.count() > 0 and await continue_button.is_visible():
            try:
                await continue_button.click(timeout=15_000, no_wait_after=True)
                await page.wait_for_timeout(2_000)
                print("ログイン継続ボタンを自動で押しました。")
            except PlaywrightTimeoutError:
                print("継続ボタンはブラウザ上で手動クリックしてください。")

        print(
            "ブラウザで『ログインの方はこちら』等のボタン、追加認証を完了し、"
            "読売新聞トップまたは記事ページまで進んでください。"
        )
        while True:
            confirmation = input(
                "ブラウザでログイン完了を確認したら DONE と入力してください: "
            ).strip().upper()
            if confirmation != "DONE":
                print("まだブラウザは閉じません。完了後に DONE と入力してください。")
                continue
            if "/member/login" in page.url:
                print(
                    f"まだログイン関連ページです: {page.url}\n"
                    "ブラウザ側のボタン操作を完了してから、もう一度 DONE と入力してください。"
                )
                continue
            break

        cookies = await context.cookies(BASE_URL)
        for cookie in cookies:
            requests_session.cookies.set(
                cookie["name"], cookie["value"],
                domain=cookie.get("domain"), path=cookie.get("path", "/"),
            )
        await browser.close()

    if not cookies:
        raise RuntimeError("認証Cookieを取得できませんでした。")
    print("読売IDのログインCookieを取得しました。")


await login_to_yomiuri(session, YOMIURI_ID, YOMIURI_PASSWORD)
del YOMIURI_ID, YOMIURI_PASSWORD


次のログイン画面へ遷移済みです: https://www.yomiuri.co.jp/
ブラウザで『ログインの方はこちら』等のボタン、追加認証を完了し、読売新聞トップまたは記事ページまで進んでください。
読売IDのログインCookieを取得しました。


In [5]:
def get_soup(url, *, params=None):
    response = session.get(url, params=params, timeout=TIMEOUT)
    response.raise_for_status()
    return BeautifulSoup(response.content, "html.parser")


def parse_search_page(soup):
    """検索結果1ページの記事、合計件数、次ページURLを返す。"""
    rows = []
    for item in soup.select(".search-article-list li.p-list-item"):
        link = item.select_one("h3.c-list-title a[href]")
        if link is None:
            continue
        time_node = item.select_one("time[datetime]")
        tags = [
            tag.get_text(" ", strip=True)
            for tag in item.select(".p-feature-tags .p-feature-tags__item")
        ]
        rows.append({
            "title": link.get_text(" ", strip=True),
            "published_at": time_node.get("datetime") if time_node else None,
            "url": urljoin(BASE_URL, link.get("href", "")),
            "tags": tags,
        })

    total_node = soup.select_one(".search-result span")
    total_count = None
    if total_node:
        count_text = total_node.get_text(strip=True).replace(",", "")
        if count_text.isdigit():
            total_count = int(count_text)

    next_node = soup.select_one(".c-pager-next a[href]")
    next_url = urljoin(SEARCH_URL, next_node.get("href", "")) if next_node else None
    return rows, total_count, next_url


def search_articles(keyword, start_date):
    """「次へ」をたどり、指定日以降の検索結果を取得する。"""
    found = []
    seen = set()
    next_url = SEARCH_URL
    params = {"st": 1, "wo": keyword, "ac": "srch", "ar": 1}
    total_count = None
    page = 0

    while True:
        page += 1
        soup = get_soup(next_url, params=params if page == 1 else None)
        page_rows, page_total, next_url = parse_search_page(soup)

        if total_count is None:
            total_count = page_total or 0
            print(f"検索結果の合計件数: {total_count:,} 件")

        new_count = 0
        reached_older_articles = False
        for row in page_rows:
            if row["url"] in seen:
                continue
            seen.add(row["url"])
            new_count += 1
            published_at = pd.to_datetime(row["published_at"], errors="coerce")
            if pd.isna(published_at):
                continue
            if published_at < start_date:
                reached_older_articles = True
                continue
            if REQUIRED_TAG in row["tags"]:
                found.append(row)

        print(
            f"検索一覧を確認中: {len(seen):,} / {total_count:,} 件 "
            f"（{start_date.date()} 以降・{REQUIRED_TAG} 一致: {len(found):,} 件）"
        )
        if reached_older_articles:
            print(f"{start_date.date()} より前の記事に達したため検索を終了します。")
            break
        if total_count and len(seen) >= total_count:
            break
        if not page_rows or new_count == 0 or not next_url:
            if total_count and len(found) < total_count:
                print(
                    f"警告: 合計 {total_count:,} 件のうち {len(seen):,} 件で"
                    "次の検索結果がなくなりました。"
                )
            break
        time.sleep(REQUEST_INTERVAL)

    print(f"{REQUIRED_TAG} が付いた記事: {len(found):,} 件")
    return found


def news_article_json_ld(soup):
    for script in soup.select('script[type="application/ld+json"]'):
        try:
            data = json.loads(script.string or script.get_text())
        except (json.JSONDecodeError, TypeError):
            continue
        candidates = data if isinstance(data, list) else [data]
        for candidate in candidates:
            if isinstance(candidate, dict) and candidate.get("@type") in {
                "NewsArticle", "Article"
            }:
                return candidate
    return {}


def parse_article_page(soup, search_row):
    metadata = news_article_json_ld(soup)
    paragraphs = []
    for node in soup.select('[itemprop="articleBody"]'):
        text = node.get_text(" ", strip=True)
        if text:
            paragraphs.append(text)

    body = "\n\n".join(dict.fromkeys(paragraphs))
    paywall = soup.select_one(
        ".p-main-contents-member-only-v2, .p-main-contents-member-only"
    )
    body_is_excerpt = paywall is not None

    if not body:
        body = str(metadata.get("description") or "").strip()
        body_is_excerpt = bool(body)

    return {
        "title": metadata.get("headline") or search_row["title"],
        "body": body,
        "published_at": (
            metadata.get("datePublished") or search_row["published_at"]
        ),
        "url": search_row["url"],
        "body_is_excerpt": body_is_excerpt,
    }


def collect_articles(keyword, start_date):
    search_rows = search_articles(keyword, start_date)
    print(f"検索結果から {len(search_rows):,} 件のURLを取得しました。")

    articles = []
    for index, row in enumerate(search_rows, start=1):
        print(f"[{index:,}/{len(search_rows):,}] {row['title']}")
        try:
            soup = get_soup(row["url"])
            articles.append(parse_article_page(soup, row))
        except requests.RequestException as exc:
            articles.append({
                **row,
                "body": "",
                "body_is_excerpt": False,
                "error": str(exc),
            })
        if index < len(search_rows):
            time.sleep(REQUEST_INTERVAL)

    return articles


In [6]:
articles = collect_articles(KEYWORD, START_DATE)

df = pd.DataFrame(articles)
if not df.empty:
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    df = df.sort_values("published_at", ascending=False, na_position="last")
df = df.reset_index(drop=True)
df.insert(0, "ID", range(1, len(df) + 1))

df


検索結果の合計件数: 4,388 件
検索一覧を確認中: 20 / 4,388 件 （#能登半島地震 一致: 1 件）
検索一覧を確認中: 40 / 4,388 件 （#能登半島地震 一致: 2 件）
検索一覧を確認中: 60 / 4,388 件 （#能登半島地震 一致: 6 件）
検索一覧を確認中: 80 / 4,388 件 （#能登半島地震 一致: 15 件）
検索一覧を確認中: 100 / 4,388 件 （#能登半島地震 一致: 20 件）
検索一覧を確認中: 120 / 4,388 件 （#能登半島地震 一致: 22 件）
検索一覧を確認中: 140 / 4,388 件 （#能登半島地震 一致: 29 件）
検索一覧を確認中: 160 / 4,388 件 （#能登半島地震 一致: 30 件）
検索一覧を確認中: 180 / 4,388 件 （#能登半島地震 一致: 34 件）
検索一覧を確認中: 200 / 4,388 件 （#能登半島地震 一致: 39 件）
検索一覧を確認中: 220 / 4,388 件 （#能登半島地震 一致: 43 件）
検索一覧を確認中: 240 / 4,388 件 （#能登半島地震 一致: 46 件）
検索一覧を確認中: 260 / 4,388 件 （#能登半島地震 一致: 48 件）
検索一覧を確認中: 280 / 4,388 件 （#能登半島地震 一致: 52 件）
検索一覧を確認中: 300 / 4,388 件 （#能登半島地震 一致: 56 件）
検索一覧を確認中: 320 / 4,388 件 （#能登半島地震 一致: 61 件）
検索一覧を確認中: 340 / 4,388 件 （#能登半島地震 一致: 63 件）
検索一覧を確認中: 360 / 4,388 件 （#能登半島地震 一致: 63 件）
検索一覧を確認中: 380 / 4,388 件 （#能登半島地震 一致: 63 件）
検索一覧を確認中: 400 / 4,388 件 （#能登半島地震 一致: 64 件）
検索一覧を確認中: 420 / 4,388 件 （#能登半島地震 一致: 65 件）
検索一覧を確認中: 440 / 4,388 件 （#能登半島地震 一致: 68 件）
検索一覧を確認中: 460 / 4,388 件 （#能登半島地震 一致: 72 件）

,ID,title,body,published_at,url,body_is_excerpt
0,1,初の復興住宅 七尾に １４戸、来月入居開始…能登地震,【読売新聞】,2026-07-24 05:00:00+09:00,https://www.yomiuri.co.jp/national/20260723-GY...,True
1,2,石川県七尾市:能登半島地震「復興住宅」１４戸完成、８月入居開始…「恒久的住まい移行という希望...,能登半島地震の被災地で初となる災害公営住宅（復興住宅）が石川県七尾市に完成し、落成式と内覧会...,2026-07-23 13:50:00+09:00,https://www.yomiuri.co.jp/national/20260723-GY...,False
2,3,石川県輪島市:世界初「ポケモン」冠した空港、巨大ピカチュウがお出迎え…「のと里山ポケモン・ウ...,能登半島地震で大きな被害を受けた石川県輪島市の能登空港（のと里山空港）が７日、「のと里山ポケ...,2026-07-07 13:26:00+09:00,https://www.yomiuri.co.jp/national/20260707-GY...,False
3,4,２度被災した石川・輪島の美術家、「作品を作ることで自分を奮い立たせ」…がれきとなった土蔵の土...,能登半島を襲った地震、豪雨災害で被災した石川県輪島市の美術家、畠中陽一さん（６９）が、岐阜市...,2026-07-07 09:57:00+09:00,https://www.yomiuri.co.jp/national/20260629-GY...,False
4,5,［のと探訪］交流増 駒に願い,【読売新聞】,2026-07-04 15:00:00+09:00,https://www.yomiuri.co.jp/national/20260704-GY...,True
...,...,...,...,...,...,...
1924,1925,セブン―イレブン、石川・富山などで１５０店臨時休業…北陸地方の広範囲で配送見合わせ,１日の能登半島地震を受け、セブン＆アイ・ホールディングスによると同日午後８時現在、石川県、富...,2024-01-01 19:11:00+09:00,https://www.yomiuri.co.jp/economy/20240101-OYT...,False
1925,1926,ジェットスター、２日からスト全面解除へ…能登半島地震の発生受け,格安航空会社（ＬＣＣ）ジェットスター・ジャパンの労働組合は１日、石川県能登地方を震源とする地...,2024-01-01 19:05:00+09:00,https://www.yomiuri.co.jp/economy/20240101-OYT...,False
1926,1927,石川県での地震、気象庁が「令和６年能登半島地震」と命名,石川県志賀町で震度７を観測した地震ついて、気象庁は１日、「令和６年能登半島地震」と命名した。,2024-01-01 18:44:00+09:00,https://www.yomiuri.co.jp/national/20240101-OY...,False
1927,1928,上越新幹線の越後湯沢―新潟と北陸新幹線の長野―金沢、１日は終日運休…能登半島地震,１日午後４時１０分頃に発生した石川県能登地方を震源とする最大震度７の地震の影響で、ＪＲ東日本...,2024-01-01 16:46:00+09:00,https://www.yomiuri.co.jp/national/20240101-OY...,False


In [7]:
output_path = Path("yomiuri_熊本地震_20260727以降.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"保存しました: {output_path.resolve()}")


保存しました: /Users/tj/IdeaProjects/sample/get-news/yomiuri_能登半島地震.csv
